# 🏏 IPL — Batter Performance vs Spin Bowling
## Enhanced ML Prediction Model — v3

### What's new in v3 vs v2:
| Feature | v2 | v3 |
|---|---|---|
| Models | RF + GBM | RF + GBM + **XGBoost + LightGBM** |
| Features | Basic batter stats | + **Venue, Recent Form, Ball-level** |
| Clustering | ❌ | ✅ **KMeans batter archetypes** |
| Hyperparameter tuning | ❌ | ✅ **RandomizedSearchCV** |
| Combined metric | ❌ | ✅ **Expected Runs** |
| Monte Carlo simulation | ❌ | ✅ **1000-iteration simulation** |
| Explainability | Feature importance only | + **SHAP values** |
| Web scraping | ❌ | ✅ **ESPNcricinfo scraper** |
| Code structure | Linear cells | **Modular functions/classes** |

---

## 📦 Step 1 — Imports

In [1]:
# ── Standard library ─────────────────────────────────────────────
import json, glob, os, warnings, time
warnings.filterwarnings('ignore')

# ── Data ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import mean_absolute_error, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.inspection import permutation_importance
import joblib


# ── NEW: XGBoost & LightGBM ──────────────────────────────────────
import xgboost as xgb
import lightgbm as lgb

# ── NEW: SHAP for explainability ─────────────────────────────────
import shap

# ── NEW: Web scraping ────────────────────────────────────────────
import urllib.request
import re
from html.parser import HTMLParser

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13

print('✅ All libraries loaded')
print(f'   XGBoost  : {xgb.__version__}')
print(f'   LightGBM : {lgb.__version__}')
print(f'   SHAP     : {shap.__version__}')

ModuleNotFoundError: No module named 'xgboost'

---
## 📂 Step 2 — Load All Data

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA_FOLDER = '.'                   # folder with CSVs and notebook
JSON_FOLDER = '../json/ipl_match'   # folder with all Cricsheet JSONs
SKIP_JSON   = {'teams_info.json'}

SPIN_TYPES = [
    'right-arm offbreak', 'legbreak', 'legbreak googly',
    'slow left-arm orthodox', 'left-arm wrist-spin'
]

# ── Load CSVs ────────────────────────────────────────────────────
balls   = pd.read_csv(os.path.join(DATA_FOLDER, 'Ball_By_Ball_Match_Data.csv'))
players = pd.read_csv(os.path.join(DATA_FOLDER, '2024_players_details.csv'))
matches = pd.read_csv(os.path.join(DATA_FOLDER, 'Match_Info.csv'))

print(f'Ball_By_Ball : {len(balls):,} rows')
print(f'Players      : {len(players):,} rows')
print(f'Matches      : {len(matches):,} rows')

In [ ]:
# ── Load all Cricsheet JSON files ────────────────────────────────
def parse_cricsheet(filepath):
    """Parse one Cricsheet JSON → list of delivery dicts."""
    match_id = os.path.splitext(os.path.basename(filepath))[0]
    rows = []
    try:
        with open(filepath, encoding='utf-8') as f:
            m = json.load(f)
    except Exception as e:
        print(f'  ⚠ Could not read {filepath}: {e}')
        return rows

    info  = m.get('info', {})
    teams = info.get('teams', [])

    for inn_idx, inn in enumerate(m.get('innings', [])):
        bt = inn.get('team', '')
        for od in inn.get('overs', []):
            ov = od.get('over', -1)
            for bn, ball in enumerate(od.get('deliveries', [])):
                extras = ball.get('extras', {})
                runs   = ball.get('runs', {})
                wkts   = ball.get('wickets', [])
                rows.append({
                    'ID'              : match_id,
                    'Innings'         : inn_idx + 1,
                    'Overs'           : ov,
                    'BallNumber'      : bn + 1,
                    'Batter'          : ball.get('batter', ''),
                    'Bowler'          : ball.get('bowler', ''),
                    'BatsmanRun'      : runs.get('batter', 0),
                    'ExtrasRun'       : runs.get('extras', 0),
                    'TotalRun'        : runs.get('total', 0),
                    'IsWicketDelivery': int(bool(wkts)),
                    'Kind'            : wkts[0].get('kind', '') if wkts else '',
                    'ExtraType'       : 'wides' if 'wides' in extras else
                                        'noballs' if 'noballs' in extras else np.nan,
                    'BattingTeam'     : bt,
                    'venue'           : info.get('venue', ''),
                })
    return rows


json_files = [
    f for f in sorted(glob.glob(os.path.join(JSON_FOLDER, '*.json')))
    if os.path.basename(f) not in SKIP_JSON
]
print(f'Found {len(json_files)} Cricsheet JSON files — parsing...')

all_json_rows = []
for i, fp in enumerate(json_files):
    all_json_rows.extend(parse_cricsheet(fp))
    if (i+1) % 200 == 0 or (i+1) == len(json_files):
        print(f'  Parsed {i+1}/{len(json_files)} files — {len(all_json_rows):,} deliveries')

json_balls = pd.DataFrame(all_json_rows) if all_json_rows else pd.DataFrame()
print(f'Done. JSON deliveries: {len(json_balls):,}')

In [ ]:
# ── Merge CSV + JSON, add season & venue ─────────────────────────
COMMON_COLS = ['ID','Innings','Overs','BallNumber','Batter','Bowler',
               'BatsmanRun','ExtrasRun','TotalRun','IsWicketDelivery',
               'Kind','ExtraType','BattingTeam']

balls_clean = balls[COMMON_COLS].copy()

if not json_balls.empty:
    json_clean = json_balls[COMMON_COLS + ['venue']].copy()
    csv_ids    = set(balls_clean['ID'].astype(str))
    json_new   = json_clean[~json_clean['ID'].astype(str).isin(csv_ids)]
    all_balls  = pd.concat([balls_clean, json_new[COMMON_COLS]], ignore_index=True)
    print(f'New rows from JSON: {len(json_new):,}')
else:
    all_balls = balls_clean.copy()

# Attach season + venue from Match_Info
matches['match_date'] = pd.to_datetime(matches['match_date'])
matches['season']     = matches['match_date'].dt.year
all_balls = all_balls.merge(
    matches[['match_number','season','venue']].rename(columns={'match_number':'ID'}),
    on='ID', how='left'
)

# Team name standardisation
TEAM_ALIAS = {
    'Kings XI Punjab'             : 'Punjab Kings',
    'Delhi Daredevils'            : 'Delhi Capitals',
    'Royal Challengers Bangalore' : 'Royal Challengers Bengaluru',
    'Deccan Chargers'             : 'Sunrisers Hyderabad',
}
all_balls['BattingTeam'] = all_balls['BattingTeam'].replace(TEAM_ALIAS)
players['longBowlingStyles'] = players['longBowlingStyles'].replace('Na', np.nan)

print(f'Total master deliveries: {len(all_balls):,}')
print(f'Seasons: {sorted(all_balls["season"].dropna().astype(int).unique())}')

---
## 🌀 Step 3 — Identify Spin Deliveries

In [ ]:
spin_map = {
    r['Name']: r['longBowlingStyles']
    for _, r in players.iterrows()
    if r['longBowlingStyles'] in SPIN_TYPES
}
print(f'Spin bowlers: {len(spin_map)}')

legal = all_balls[~all_balls['ExtraType'].isin(['wides','noballs'])].copy()
spin  = legal[legal['Bowler'].isin(spin_map)].copy()
spin['spin_type'] = spin['Bowler'].map(spin_map)
spin['phase']     = pd.cut(spin['Overs'], bins=[-1,5,14,19],
                            labels=['powerplay','middle','death'])

print(f'Spin legal deliveries : {len(spin):,}')
print(f'Unique batters        : {spin["Batter"].nunique()}')

---
## ⚙️ Step 4 — Advanced Feature Engineering
# NEW: Venue features, recent form, ball-level features added

In [ ]:
# ── 4a: Batter career features vs spin (EXISTING — kept) ─────────
def build_batter_features(spin_df):
    """Compute career batting stats vs spin for every batter."""
    bf = spin_df.groupby('Batter').agg(
        total_balls      = ('BatsmanRun', 'count'),
        total_runs       = ('BatsmanRun', 'sum'),
        dismissals       = ('IsWicketDelivery', 'sum'),
        dots             = ('BatsmanRun', lambda x: (x==0).sum()),
        fours            = ('BatsmanRun', lambda x: (x==4).sum()),
        sixes            = ('BatsmanRun', lambda x: (x==6).sum()),
        ones             = ('BatsmanRun', lambda x: (x==1).sum()),
        twos             = ('BatsmanRun', lambda x: (x==2).sum()),
    ).reset_index()
    bf['sr']           = bf['total_runs'] / bf['total_balls'] * 100
    bf['avg']          = (bf['total_runs'] / bf['dismissals'].replace(0, np.nan)).fillna(bf['total_runs'])
    bf['dot_pct']      = bf['dots']  / bf['total_balls'] * 100
    bf['boundary_pct'] = (bf['fours'] + bf['sixes']) / bf['total_balls'] * 100
    bf['six_pct']      = bf['sixes'] / bf['total_balls'] * 100
    bf['wkt_rate']     = bf['dismissals'] / bf['total_balls'] * 100
    bf['rotation_pct'] = (bf['ones'] + bf['twos']) / bf['total_balls'] * 100
    return bf

batter_feats = build_batter_features(spin)
print(f'Batter features: {batter_feats.shape}')

In [ ]:
# NEW: 4b — Venue-based spin features
# How spin-friendly is each ground? (wicket rate & economy at that venue vs spin)
def build_venue_features(spin_df):
    """
    Per-venue spin stats:
    - venue_spin_wkt_rate : how often spinners take wickets here
    - venue_spin_economy  : average runs per ball off spin here
    A high wkt_rate + low economy = spin-friendly venue
    """
    vf = spin_df.groupby('venue').agg(
        venue_spin_wkt_rate = ('IsWicketDelivery', 'mean'),
        venue_spin_economy  = ('BatsmanRun', 'mean'),
        venue_deliveries    = ('BatsmanRun', 'count'),
    ).reset_index()
    # Only use venues with enough data (min 50 spin deliveries)
    vf = vf[vf['venue_deliveries'] >= 50].drop(columns='venue_deliveries')
    return vf

venue_feats = build_venue_features(spin)
print(f'Venues with enough spin data: {len(venue_feats)}')
print(venue_feats.sort_values('venue_spin_wkt_rate', ascending=False).head(5).to_string())

In [ ]:
# NEW: 4c — Recent form (last 5 innings vs spin)
# Captures whether a batter is currently in form or struggling vs spin
def build_recent_form(spin_df):
    """
    Per batter: rolling average SR across last 5 innings vs spin.
    Returns a DataFrame with Batter → form_sr_last5
    """
    # Aggregate per innings
    inn_agg = (
        spin_df.sort_values(['Batter','ID','Innings'])
        .groupby(['Batter','ID','Innings'])
        .agg(inn_runs=('BatsmanRun','sum'), inn_balls=('BatsmanRun','count'))
        .reset_index()
    )
    inn_agg['inn_sr'] = inn_agg['inn_runs'] / inn_agg['inn_balls'] * 100

    # Rolling last-5 SR (shifted so we don't leak current innings)
    inn_agg = inn_agg.sort_values(['Batter','ID'])
    inn_agg['form_sr_last5'] = (
        inn_agg.groupby('Batter')['inn_sr']
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

    # Take the most recent value per batter
    form = inn_agg.groupby('Batter')['form_sr_last5'].last().reset_index()
    return form

form_feats = build_recent_form(spin)
print(f'Recent form computed for {len(form_feats)} batters')
print(form_feats.sort_values('form_sr_last5', ascending=False).head(5))

In [ ]:
# NEW: 4d — Ball-level derived features
# Context within the over: early ball vs late ball matters vs spin
spin['ball_in_over_norm'] = spin['BallNumber'] / 6          # 0–1 normalised
spin['is_last_ball']      = (spin['BallNumber'] == 6).astype(int)
spin['over_x_ball']       = spin['Overs'] * spin['BallNumber']   # interaction term
spin['is_death_last']     = ((spin['Overs'] >= 17) & (spin['BallNumber'] >= 5)).astype(int)

print('Ball-level features added: ball_in_over_norm, is_last_ball, over_x_ball, is_death_last')

---
## 🔵 Step 5 — KMeans Batter Clustering
# NEW: Classify batters into archetypes based on spin performance

In [ ]:
# NEW: KMeans clustering — group batters into 4 archetypes vs spin
# Features used: strike rate, dot%, boundary%, wicket rate
# Why? Cluster label adds richer context to the supervised model

CLUSTER_FEATURES = ['sr', 'dot_pct', 'boundary_pct', 'wkt_rate']
CLUSTER_NAMES    = {
    0: 'Aggressive Attacker',
    1: 'Steady Accumulator',
    2: 'Spin Vulnerable',
    3: 'Balanced Performer'
}

# Only cluster batters with enough data
bf_q = batter_feats[batter_feats['total_balls'] >= 30].copy()

scaler    = StandardScaler()
X_cluster = scaler.fit_transform(bf_q[CLUSTER_FEATURES].fillna(0))

# Find optimal K using inertia (elbow method)
inertias = []
K_range  = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

# Plot elbow
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_range, inertias, 'bo-')
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method — Optimal K for Batter Clustering')
ax.axvline(4, color='red', linestyle='--', alpha=0.5, label='K=4 chosen')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# NEW: Fit final KMeans with K=4 and label each batter
N_CLUSTERS = 4
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
bf_q['cluster'] = kmeans.fit_predict(X_cluster)

# Assign cluster names by inspecting centroid characteristics
# (cluster with highest SR = Aggressive, highest wkt_rate = Vulnerable, etc.)
cluster_summary = bf_q.groupby('cluster')[CLUSTER_FEATURES].mean().round(2)
print('Cluster Centroids:')
print(cluster_summary)
print(f'\nBatters per cluster: {bf_q["cluster"].value_counts().to_dict()}')

# Visualise clusters
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#2ecc71','#3498db','#e74c3c','#f39c12']
for c in range(N_CLUSTERS):
    mask = bf_q['cluster'] == c
    ax.scatter(bf_q[mask]['dot_pct'], bf_q[mask]['sr'],
               s=60, alpha=0.7, color=colors[c], label=f'Cluster {c}')

ax.set_xlabel('Dot Ball % vs Spin')
ax.set_ylabel('Strike Rate vs Spin')
ax.set_title('KMeans Batter Clusters — Strike Rate vs Dot Ball %')
ax.legend()

# Annotate a few top batters
for _, row in bf_q.nlargest(8,'total_balls').iterrows():
    ax.annotate(row['Batter'], (row['dot_pct'], row['sr']), fontsize=7, alpha=0.85)

plt.tight_layout()
plt.show()

# Merge cluster labels back into batter_feats for use in supervised model
batter_feats = batter_feats.merge(
    bf_q[['Batter','cluster']], on='Batter', how='left'
)
batter_feats['cluster'] = batter_feats['cluster'].fillna(-1).astype(int)
print('\nCluster labels added to batter_feats ✅')

---
## ⚙️ Step 6 — Build Model Dataset

In [ ]:
# Encode categoricals
le_spin  = LabelEncoder()
le_phase = LabelEncoder()
spin['spin_type_enc'] = le_spin.fit_transform(spin['spin_type'].astype(str))
spin['phase_enc']     = le_phase.fit_transform(spin['phase'].astype(str))

# UPDATED: expanded feature set including all new features
BATTER_COLS = ['Batter','sr','avg','dot_pct','boundary_pct',
               'six_pct','wkt_rate','rotation_pct','cluster']

model_df = spin.merge(batter_feats[BATTER_COLS].fillna(0), on='Batter', how='left')
model_df = model_df.merge(venue_feats, on='venue', how='left')
model_df = model_df.merge(form_feats, on='Batter', how='left')

# Fill venue/form nulls with global mean
model_df['venue_spin_wkt_rate'] = model_df['venue_spin_wkt_rate'].fillna(
    model_df['venue_spin_wkt_rate'].mean())
model_df['venue_spin_economy']  = model_df['venue_spin_economy'].fillna(
    model_df['venue_spin_economy'].mean())
model_df['form_sr_last5']       = model_df['form_sr_last5'].fillna(model_df['sr'])

# UPDATED: full feature list including NEW features
FEATURES = [
    # Delivery context (EXISTING)
    'Overs', 'BallNumber', 'Innings',
    'spin_type_enc', 'phase_enc',
    # NEW: ball-level derived
    'ball_in_over_norm', 'is_last_ball', 'over_x_ball', 'is_death_last',
    # Batter career stats (EXISTING)
    'sr', 'avg', 'dot_pct', 'boundary_pct',
    'six_pct', 'wkt_rate', 'rotation_pct',
    # NEW: cluster archetype
    'cluster',
    # NEW: venue features
    'venue_spin_wkt_rate', 'venue_spin_economy',
    # NEW: recent form
    'form_sr_last5',
]

model_df = model_df[FEATURES + ['BatsmanRun','IsWicketDelivery']].dropna()
print(f'Model dataset: {len(model_df):,} rows × {len(FEATURES)} features')
print(f'Wicket rate  : {model_df["IsWicketDelivery"].mean()*100:.2f}%')

In [ ]:
# Train / Test Split
X  = model_df[FEATURES]
yr = model_df['BatsmanRun']
yw = model_df['IsWicketDelivery']

X_tr, X_te, yr_tr, yr_te, yw_tr, yw_te = train_test_split(
    X, yr, yw, test_size=0.2, random_state=42
)
print(f'Train: {len(X_tr):,}  |  Test: {len(X_te):,}')

---
## 🧠 Step 7 — Train & Compare All Models
# NEW: XGBoost and LightGBM added alongside existing RF and GBM

In [ ]:
# NEW: Model registry — train all 4 models and compare
# Each model is trained for both tasks: runs (regression) and wicket (classification)

print('Training all models...\n')

# ── RUNS MODELS (Regressor) ──────────────────────────────────────
runs_models = {
    'RandomForest': RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=20,
        random_state=42, n_jobs=-1),

    # NEW
    'XGBoost': xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0),

    # NEW
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1),
}

# ── WICKET MODELS (Classifier) ───────────────────────────────────
wkt_models = {
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42),

    # NEW
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, scale_pos_weight=10,
        random_state=42, verbosity=0),

    # NEW
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, class_weight='balanced',
        random_state=42, verbose=-1),
}

# ── Train and evaluate all ───────────────────────────────────────
runs_results = {}
for name, m in runs_models.items():
    m.fit(X_tr, yr_tr)
    mae = mean_absolute_error(yr_te, m.predict(X_te))
    runs_results[name] = mae
    print(f'  Runs  [{name:18s}] MAE : {mae:.4f}')

print()
wkt_results = {}
for name, m in wkt_models.items():
    m.fit(X_tr, yw_tr)
    auc = roc_auc_score(yw_te, m.predict_proba(X_te)[:,1])
    wkt_results[name] = auc
    print(f'  Wicket [{name:18s}] AUC : {auc:.4f}')

# Select best model for each task
best_runs_name = min(runs_results, key=runs_results.get)
best_wkt_name  = max(wkt_results,  key=wkt_results.get)
print(f'\n🏆 Best Runs model   : {best_runs_name} (MAE={runs_results[best_runs_name]:.4f})')
print(f'🏆 Best Wicket model : {best_wkt_name}  (AUC={wkt_results[best_wkt_name]:.4f})')

best_runs_model = runs_models[best_runs_name]
best_wkt_model  = wkt_models[best_wkt_name]

In [ ]:
# Model comparison chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(runs_results.keys(), runs_results.values(), color=['steelblue','#e67e22','#2ecc71'])
axes[0].set_title('Runs Model — MAE Comparison (lower = better)')
axes[0].set_ylabel('MAE')
axes[0].set_ylim(min(runs_results.values())*0.99, max(runs_results.values())*1.01)

axes[1].bar(wkt_results.keys(), wkt_results.values(), color=['steelblue','#e67e22','#2ecc71'])
axes[1].set_title('Wicket Model — ROC-AUC Comparison (higher = better)')
axes[1].set_ylabel('AUC')
axes[1].set_ylim(min(wkt_results.values())*0.99, max(wkt_results.values())*1.01)

plt.tight_layout()
plt.show()

---
## 🔧 Step 8 — Hyperparameter Tuning
# NEW: RandomizedSearchCV on best model

In [ ]:
# NEW: Hyperparameter tuning using RandomizedSearchCV
# Uses random sampling — much faster than GridSearchCV, nearly as good
# We tune the wicket model since AUC has most room to improve

print('Tuning XGBoost Wicket model with RandomizedSearchCV...')
print('(This takes ~2-3 mins — reduce n_iter to 10 if too slow)\n')

# NEW: Parameter search space
param_dist_wkt = {
    'n_estimators'    : [100, 200, 300, 500],
    'max_depth'       : [3, 4, 5, 6],
    'learning_rate'   : [0.01, 0.03, 0.05, 0.1],
    'subsample'       : [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'scale_pos_weight': [5, 10, 15, 20],
}

xgb_base = xgb.XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss')

rs_wkt = RandomizedSearchCV(
    xgb_base,
    param_distributions = param_dist_wkt,
    n_iter              = 20,          # increase to 50+ for better tuning
    scoring             = 'roc_auc',
    cv                  = StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state        = 42,
    n_jobs              = -1,
    verbose             = 1,
)
rs_wkt.fit(X_tr, yw_tr)

tuned_wkt_model  = rs_wkt.best_estimator_
tuned_auc        = roc_auc_score(yw_te, tuned_wkt_model.predict_proba(X_te)[:,1])

print(f'\nBest params : {rs_wkt.best_params_}')
print(f'Before tuning AUC : {wkt_results["XGBoost"]:.4f}')
print(f'After tuning AUC  : {tuned_auc:.4f}')

# Use tuned model going forward
best_wkt_model = tuned_wkt_model

In [ ]:
# Similarly tune the Runs model
print('Tuning LightGBM Runs model...')

param_dist_runs = {
    'n_estimators'    : [100, 200, 300, 500],
    'max_depth'       : [4, 6, 8, 10],
    'learning_rate'   : [0.01, 0.03, 0.05, 0.1],
    'subsample'       : [0.6, 0.8, 0.9, 1.0],
    'num_leaves'      : [15, 31, 63, 127],
}

lgb_base = lgb.LGBMRegressor(random_state=42, verbose=-1)

rs_runs = RandomizedSearchCV(
    lgb_base,
    param_distributions = param_dist_runs,
    n_iter              = 20,
    scoring             = 'neg_mean_absolute_error',
    cv                  = 3,
    random_state        = 42,
    n_jobs              = -1,
    verbose             = 1,
)
rs_runs.fit(X_tr, yr_tr)

tuned_runs_model = rs_runs.best_estimator_
tuned_mae        = mean_absolute_error(yr_te, tuned_runs_model.predict(X_te))

print(f'\nBefore tuning MAE : {runs_results["LightGBM"]:.4f}')
print(f'After tuning MAE  : {tuned_mae:.4f}')

best_runs_model = tuned_runs_model

---
## 💡 Step 9 — Model Explainability with SHAP
# NEW: SHAP values show WHY the model made each prediction

In [ ]:
# NEW: SHAP explainability for the Wicket model
# SHAP (SHapley Additive exPlanations) tells us:
#   - Which features push the prediction UP (towards wicket)
#   - Which features push it DOWN (away from wicket)
#   - How much each feature matters globally

print('Computing SHAP values (may take ~30 seconds)...')

explainer_wkt  = shap.TreeExplainer(best_wkt_model)
shap_values_wkt = explainer_wkt.shap_values(X_te.iloc[:1000])  # sample for speed

# Global feature importance — beeswarm plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_wkt, X_te.iloc[:1000],
                  feature_names=FEATURES,
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Wicket Model')
plt.tight_layout()
plt.show()

In [ ]:
# NEW: SHAP beeswarm — shows direction of each feature's effect
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values_wkt, X_te.iloc[:1000],
                  feature_names=FEATURES, show=False)
plt.title('SHAP Beeswarm — Wicket Model\n(Red = high feature value, Blue = low)')
plt.tight_layout()
plt.show()

In [ ]:
# NEW: SHAP for Runs model
explainer_runs  = shap.TreeExplainer(best_runs_model)
shap_values_runs = explainer_runs.shap_values(X_te.iloc[:1000])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_runs, X_te.iloc[:1000],
                  feature_names=FEATURES, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Runs Model')
plt.tight_layout()
plt.show()

---
## 🌐 Step 10 — Web Scraping (ESPNcricinfo)
# NEW: Scrape upcoming match info — teams, venue, playing XI

In [ ]:
# NEW: ESPNcricinfo scraper
# Fetches upcoming IPL match details: teams, venue, playing XI
# Note: If ESPN blocks the request, use the manual_match_input() fallback below

class CricinfoScraper:
    """Scrape live/upcoming IPL match data from ESPNcricinfo."""

    HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) '
                      'Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
    }

    def fetch_url(self, url, timeout=10):
        """Fetch a URL and return HTML text, or None on failure."""
        try:
            req = urllib.request.Request(url, headers=self.HEADERS)
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read().decode('utf-8', errors='ignore')
        except Exception as e:
            print(f'  ⚠ Fetch failed: {e}')
            return None

    def get_live_matches(self):
        """
        Fetch current live IPL matches.
        Returns list of dicts: {match_id, description, teams, venue, status}
        """
        html = self.fetch_url('https://www.espncricinfo.com/cricket-match/live-cricket-scores')
        if not html:
            return []

        matches = []
        # Extract match blocks using regex on the JSON-LD or script tags
        ids = re.findall(r'match/(\d+)/', html)
        teams_raw = re.findall(r'([A-Z][a-z]+ (?:[A-Z][a-z]+ ?)+) vs ([A-Z][a-z]+ (?:[A-Z][a-z]+ ?)+)', html)
        for i, mid in enumerate(set(ids[:10])):
            matches.append({'match_id': mid, 'teams': teams_raw[i] if i < len(teams_raw) else ('TBA','TBA')})
        return matches

    def get_playing_xi(self, match_url):
        """
        Given an ESPN match URL, extract playing XI for both teams.
        Returns dict: {team1_name: [players], team2_name: [players], venue: str}
        """
        html = self.fetch_url(match_url)
        if not html:
            return None

        result = {'venue': '', 'team1': '', 'team2': '', 'team1_xi': [], 'team2_xi': []}

        # Extract venue
        venue_match = re.search(r'Venue[:\s]+([^<\n,]+)', html)
        if venue_match:
            result['venue'] = venue_match.group(1).strip()

        # Extract team names
        team_match = re.findall(r'<title>([^|<]+) vs ([^|<]+)\|', html)
        if team_match:
            result['team1'] = team_match[0][0].strip()
            result['team2'] = team_match[0][1].strip()

        # Extract player names from squads section
        players_raw = re.findall(r'data-player-id="\d+"[^>]*>([A-Z][a-z]+ [A-Z][a-z]+)', html)
        if players_raw:
            half = len(players_raw) // 2
            result['team1_xi'] = players_raw[:half]
            result['team2_xi'] = players_raw[half:]

        return result


def manual_match_input():
    """
    Fallback: manually define match data when scraping is blocked.
    Edit the values below to match the upcoming game.
    """
    return {
        'venue'   : 'Wankhede Stadium',          # ← change
        'team1'   : 'Mumbai Indians',             # ← change
        'team2'   : 'Chennai Super Kings',        # ← change
        'team1_xi': ['RG Sharma', 'IS Kishan', 'SA Yadav', 'TU Deshpande',
                     'HH Pandya', 'RA Jadeja', 'KA Pollard', 'JJ Bumrah',
                     'T Boult', 'DN Patel', 'KH Pandya'],
        'team2_xi': ['MS Dhoni', 'RD Gaikwad', 'Shivam Dube', 'AT Rayudu',
                     'MM Ali', 'DP Conway', 'M Theekshana', 'TU Deshpande',
                     'DS Kulkarni', 'MJ Santner', 'SB Dube'],
    }


# Try scraping — fall back to manual if it fails
scraper = CricinfoScraper()
print('Attempting to scrape ESPNcricinfo...')
live = scraper.get_live_matches()

if live:
    print(f'Found {len(live)} live/upcoming matches')
    upcoming_match = live[0]
else:
    print('Scraping blocked or no live matches — using manual input')
    upcoming_match = manual_match_input()
    print(f"Match: {upcoming_match['team1']} vs {upcoming_match['team2']}")
    print(f"Venue: {upcoming_match['venue']}")

---
## 🔮 Step 11 — Full Prediction System
# UPDATED: Now includes Expected Runs, cluster context, venue adjustment

In [ ]:
# UPDATED: predict_batter now includes Expected Runs = runs * (1 - dismissal_prob)
# and uses all new features

def predict_batter(batter_name, spin_type='right-arm offbreak',
                   phase='middle', innings=1, n_balls=12,
                   venue=None, verbose=True):
    """
    Predict a batter's performance against a spin type.

    Parameters
    ----------
    batter_name : str
    spin_type   : str  — one of SPIN_TYPES
    phase       : str  — 'powerplay', 'middle', 'death'
    innings     : int  — 1 or 2
    n_balls     : int  — number of balls to project over
    venue       : str  — venue name (optional, improves prediction)

    Returns dict with all prediction metrics.
    """
    # ── Batter features ──────────────────────────────────────────
    brow = batter_feats[batter_feats['Batter'] == batter_name]
    if brow.empty:
        if verbose: print(f"⚠  '{batter_name}' not found — using dataset average")
        brow = batter_feats[batter_feats['total_balls'] >= 30].mean(numeric_only=True)
        brow = brow.to_frame().T
        confidence = 'LOW (not in training data)'
    else:
        nb = int(brow['total_balls'].values[0])
        confidence = 'HIGH' if nb >= 100 else 'MEDIUM' if nb >= 30 else 'LOW (<30 balls)'
        if verbose: print(f'📊 Historical: {nb} balls vs spin | Cluster: {int(brow["cluster"].values[0])}')

    # ── Encode ───────────────────────────────────────────────────
    spin_enc  = le_spin.transform([spin_type])[0] if spin_type in le_spin.classes_ else 0
    phase_enc = le_phase.transform([phase])[0]    if phase  in le_phase.classes_  else 1

    # ── Venue lookup ─────────────────────────────────────────────
    vrow = venue_feats[venue_feats['venue'] == venue] if venue else pd.DataFrame()
    v_wkt = float(vrow['venue_spin_wkt_rate'].values[0]) if not vrow.empty \
            else float(venue_feats['venue_spin_wkt_rate'].mean())
    v_eco = float(vrow['venue_spin_economy'].values[0])  if not vrow.empty \
            else float(venue_feats['venue_spin_economy'].mean())

    # ── Form ─────────────────────────────────────────────────────
    form_row = form_feats[form_feats['Batter'] == batter_name]
    form_sr  = float(form_row['form_sr_last5'].values[0]) if not form_row.empty \
               else float(brow['sr'].values[0])

    # ── Build feature vector ─────────────────────────────────────
    over_map  = {'powerplay': 3, 'middle': 10, 'death': 17}
    over_num  = over_map[phase]
    ball_num  = 3

    feat_vals = [
        over_num, ball_num, innings,
        spin_enc, phase_enc,
        ball_num / 6,                              # ball_in_over_norm
        0,                                         # is_last_ball
        over_num * ball_num,                       # over_x_ball
        int(over_num >= 17 and ball_num >= 5),     # is_death_last
        float(brow['sr'].values[0]),
        float(brow['avg'].values[0]),
        float(brow['dot_pct'].values[0]),
        float(brow['boundary_pct'].values[0]),
        float(brow['six_pct'].values[0]),
        float(brow['wkt_rate'].values[0]),
        float(brow['rotation_pct'].values[0]),
        float(brow['cluster'].values[0]),
        v_wkt, v_eco,
        form_sr,
    ]

    X_pred = pd.DataFrame([feat_vals], columns=FEATURES)

    # ── Predictions ──────────────────────────────────────────────
    pred_runs_pb   = max(0, float(best_runs_model.predict(X_pred)[0]))
    pred_dism_prob = float(best_wkt_model.predict_proba(X_pred)[0][1])

    pred_sr           = round(pred_runs_pb * 100, 1)
    pred_runs_total   = round(pred_runs_pb * n_balls, 1)
    dismissal_pct     = round(pred_dism_prob * 100, 2)
    dismiss_in_spell  = round((1 - (1 - pred_dism_prob) ** n_balls) * 100, 1)

    # NEW: Expected Runs = runs adjusted for dismissal risk
    expected_runs = round(pred_runs_total * (1 - pred_dism_prob), 1)

    if verbose:
        print()
        print('═' * 58)
        print(f'  🏏 {batter_name}')
        print(f'  vs {spin_type}  |  {phase}  |  inn {innings}  |  venue: {venue or "unknown"}')
        print('═' * 58)
        print(f'  Predicted Strike Rate       : {pred_sr}')
        print(f'  Predicted Runs ({n_balls} balls)    : {pred_runs_total}')
        print(f'  Dismissal prob / ball       : {dismissal_pct}%')
        print(f'  Dismissal prob in spell     : {dismiss_in_spell}%')
        print(f'  ★ Expected Runs (adj. risk) : {expected_runs}')  # NEW
        print(f'  Confidence                  : {confidence}')
        print(f'  Historical SR vs spin       : {round(float(brow["sr"].values[0]),1)}')
        print('═' * 58)

    return {
        'batter'              : batter_name,
        'spin_type'           : spin_type,
        'phase'               : phase,
        'predicted_sr'        : pred_sr,
        'predicted_runs'      : pred_runs_total,
        'expected_runs'       : expected_runs,          # NEW
        'dismissal_prob_pct'  : dismissal_pct,
        'dismiss_in_spell_pct': dismiss_in_spell,
        'confidence'          : confidence,
    }


print('✅ predict_batter() ready')

In [ ]:
# ── Example prediction ────────────────────────────────────────────
result = predict_batter(
    batter_name = 'V Kohli',
    spin_type   = 'legbreak googly',
    phase       = 'middle',
    innings     = 1,
    n_balls     = 12,
    venue       = 'Wankhede Stadium'
)

---
## 🎲 Step 12 — Monte Carlo Simulation
# NEW: 1000-iteration simulation to estimate probability distribution of outcomes

In [ ]:
# NEW: Monte Carlo simulation
# Instead of a single point prediction, simulate 1000 possible outcomes
# This gives us a range: best case, worst case, most likely

def monte_carlo_simulate(batter_name, spin_type, phase, innings=1,
                          n_balls=12, n_simulations=1000, venue=None):
    """
    Monte Carlo simulation of batter performance.

    Each simulation:
    - On each ball: sample runs from model distribution
    - With dismissal probability: randomly terminate the innings
    Returns distribution of total runs across n_simulations.
    """
    # Get base prediction
    pred = predict_batter(batter_name, spin_type, phase, innings,
                          n_balls, venue, verbose=False)

    pred_runs_pb  = pred['predicted_runs'] / n_balls
    dism_prob     = pred['dismissal_prob_pct'] / 100

    # Historical run distribution for this batter (to sample from)
    batter_balls = spin[spin['Batter'] == batter_name]['BatsmanRun']
    if len(batter_balls) < 20:
        # Fall back to global distribution
        batter_balls = spin['BatsmanRun']

    run_distribution = batter_balls.values

    # Simulate
    np.random.seed(42)
    simulation_totals = []

    for _ in range(n_simulations):
        total = 0
        dismissed = False
        for ball in range(n_balls):
            if np.random.random() < dism_prob:
                dismissed = True
                break
            # Sample from actual historical run distribution
            total += np.random.choice(run_distribution)
        simulation_totals.append(total)

    totals = np.array(simulation_totals)

    # Results
    print(f'\n🎲 Monte Carlo Simulation — {batter_name} vs {spin_type}')
    print(f'   Balls: {n_balls}  |  Simulations: {n_simulations:,}')
    print(f'   ─────────────────────────────────────────')
    print(f'   Mean runs           : {totals.mean():.1f}')
    print(f'   Median runs         : {np.median(totals):.1f}')
    print(f'   Worst  10% (P10)    : {np.percentile(totals,10):.1f}')
    print(f'   Best   10% (P90)    : {np.percentile(totals,90):.1f}')
    print(f'   P(≥15 runs)         : {(totals>=15).mean()*100:.1f}%')
    print(f'   P(dismissed)        : {(totals==0).mean()*100:.1f}%')

    # Plot distribution
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(totals, bins=30, color='steelblue', alpha=0.75, edgecolor='white')
    ax.axvline(totals.mean(),        color='red',   linestyle='--', label=f'Mean={totals.mean():.1f}')
    ax.axvline(np.percentile(totals,10), color='orange', linestyle=':', label=f'P10={np.percentile(totals,10):.1f}')
    ax.axvline(np.percentile(totals,90), color='green',  linestyle=':', label=f'P90={np.percentile(totals,90):.1f}')
    ax.set_xlabel(f'Total Runs in {n_balls} Balls')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Monte Carlo: {batter_name} vs {spin_type} ({phase}, {n_simulations:,} simulations)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return totals


# ── Run a simulation ─────────────────────────────────────────────
sim_results = monte_carlo_simulate(
    batter_name   = 'V Kohli',           # ← change batter
    spin_type     = 'legbreak googly',   # ← change spin type
    phase         = 'middle',
    n_balls       = 12,
    n_simulations = 1000
)

---
## 🏟️ Step 13 — Full Match Prediction System
# NEW: Predict all batters in a playing XI against spin bowlers in that team

In [ ]:
# NEW: Full match prediction function
# Given two teams and a venue, predict how each batter will fare vs spin

def predict_match_vs_spin(batting_xi, bowling_team_xi, venue, phase='middle', n_balls=12):
    """
    Predict all batters in batting_xi against spin bowlers in bowling_team_xi.

    Parameters
    ----------
    batting_xi     : list of batter names
    bowling_team_xi: list of bowler names (spins are auto-identified)
    venue          : str
    phase          : str
    n_balls        : int

    Returns DataFrame sorted by expected_runs.
    """
    # Find spin bowlers in the bowling team
    spin_bowlers_in_team = [
        (b, spin_map[b]) for b in bowling_team_xi if b in spin_map
    ]

    if not spin_bowlers_in_team:
        print('⚠ No spin bowlers found in bowling XI')
        return pd.DataFrame()

    print(f'Spin bowlers identified in bowling team:')
    for b, t in spin_bowlers_in_team: print(f'  {b} — {t}')
    print()

    # Use most common spin type in the team for prediction
    primary_spin = spin_bowlers_in_team[0][1]

    results = []
    for batter in batting_xi:
        r = predict_batter(batter, spin_type=primary_spin,
                           phase=phase, n_balls=n_balls,
                           venue=venue, verbose=False)
        results.append(r)

    df = pd.DataFrame(results)
    df = df[['batter','predicted_sr','predicted_runs','expected_runs',
             'dismissal_prob_pct','dismiss_in_spell_pct','confidence']]
    df.columns = ['Batter','Pred SR','Pred Runs','Expected Runs',
                  'Dismiss%/ball','Dismiss% in spell','Confidence']
    return df.sort_values('Expected Runs', ascending=False)


# ── Use upcoming_match data ───────────────────────────────────────
print(f"=== MATCH PREDICTION ===")
print(f"{upcoming_match['team1']} batting vs {upcoming_match['team2']}")
print(f"Venue: {upcoming_match['venue']}\n")

match_pred = predict_match_vs_spin(
    batting_xi      = upcoming_match['team1_xi'],
    bowling_team_xi = upcoming_match['team2_xi'],
    venue           = upcoming_match['venue'],
    phase           = 'middle',
    n_balls         = 12
)
match_pred

In [ ]:
# Visualise match prediction
if not match_pred.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = ['#2ecc71' if x > 120 else '#f39c12' if x > 90 else '#e74c3c'
              for x in match_pred['Pred SR']]
    axes[0].barh(match_pred['Batter'], match_pred['Expected Runs'], color=colors)
    axes[0].set_title('Expected Runs vs Spin (adj. for dismissal risk)')
    axes[0].set_xlabel('Expected Runs (12 balls)')
    axes[0].invert_yaxis()

    d_colors = ['#2ecc71' if x < 6 else '#f39c12' if x < 10 else '#e74c3c'
                for x in match_pred['Dismiss%/ball']]
    axes[1].barh(match_pred['Batter'], match_pred['Dismiss%/ball'], color=d_colors)
    axes[1].set_title('Dismissal Probability per Ball (%)')
    axes[1].set_xlabel('Probability (%)')
    axes[1].invert_yaxis()

    plt.suptitle(f"{upcoming_match['team1']} vs Spin — {upcoming_match['venue']}",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 💾 Step 14 — Save Everything

In [ ]:
# Save models
joblib.dump(best_runs_model,  'model_runs.pkl')
joblib.dump(best_wkt_model,   'model_wicket.pkl')
joblib.dump(le_spin,          'encoder_spin.pkl')
joblib.dump(le_phase,         'encoder_phase.pkl')
joblib.dump(scaler,           'scaler_cluster.pkl')      # NEW
joblib.dump(kmeans,           'model_kmeans.pkl')        # NEW

# Save data
batter_feats.to_csv('batter_features.csv', index=False)
venue_feats.to_csv('venue_features.csv',   index=False)  # NEW
form_feats.to_csv('form_features.csv',     index=False)  # NEW

print('✅ Saved:')
print('  model_runs.pkl, model_wicket.pkl')
print('  encoder_spin.pkl, encoder_phase.pkl')
print('  model_kmeans.pkl, scaler_cluster.pkl   ← NEW')
print('  batter_features.csv, venue_features.csv, form_features.csv')

---
## 📋 Quick Reference

### Single batter prediction
```python
predict_batter('RG Sharma', spin_type='legbreak googly',
               phase='death', n_balls=12, venue='Wankhede Stadium')
```

### Monte Carlo simulation
```python
monte_carlo_simulate('RG Sharma', 'legbreak googly',
                     phase='middle', n_balls=12, n_simulations=1000)
```

### Full team prediction
```python
predict_match_vs_spin(
    batting_xi      = ['V Kohli', 'RG Sharma', ...],
    bowling_team_xi = ['YS Chahal', 'RA Jadeja', ...],
    venue           = 'Eden Gardens',
    phase           = 'middle'
)
```

### Install new libraries (if needed)
```bash
pip install xgboost lightgbm shap
```